In [0]:
import pandas as pd
path = "/Volumes/workspace/default/air_quality_data/aqi_silver_python.csv"
silver = pd.read_csv(path)
print("Silver rows:", len(silver))
display(silver.head())


Silver rows: 28157


City,Date,PM2_5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,AQI,AQI_Bucket,Year,Month,Season,AQI_Above_500
Ahmedabad,2015-01-01,null,null,0.92,18.22,17.15,null,0.92,27.64,133.36,0.0,0.02,null,null,2015,1,Winter,false
Ahmedabad,2015-01-02,null,null,0.97,15.69,16.46,null,0.97,24.55,34.06,3.68,5.5,null,null,2015,1,Winter,false
Ahmedabad,2015-01-03,null,null,17.4,19.3,29.7,null,17.4,29.07,30.7,6.8,16.4,null,null,2015,1,Winter,false
Ahmedabad,2015-01-04,null,null,1.7,18.48,17.97,null,1.7,18.59,36.08,4.43,10.14,null,null,2015,1,Winter,false
Ahmedabad,2015-01-05,null,null,22.1,21.42,37.76,null,22.1,39.33,39.31,7.01,18.89,null,null,2015,1,Winter,false


In [0]:
# City-wise Average AQI
gold_city_basic = (
silver[silver["AQI"].notna()]
.groupby("City")
.agg(
avg_aqi=("AQI", "mean"),
max_aqi=("AQI", "max"),
days_with_aqi=("AQI", "count")
)
.round({"avg_aqi": 1, "max_aqi": 1})
.reset_index()
.sort_values("avg_aqi", ascending=False)
)

display(gold_city_basic)

City,avg_aqi,max_aqi,days_with_aqi
Ahmedabad,452.1,2049.0,1334
Delhi,259.5,716.0,1999
Patna,240.8,619.0,1459
Gurugram,225.1,891.0,1453
Lucknow,218.0,707.0,1893
Talcher,172.9,570.0,698
Jorapokhar,159.3,604.0,771
Brajrajnagar,150.3,355.0,713
Kolkata,140.6,475.0,754
Guwahati,140.1,956.0,495


In [0]:
# Ahmedabas year wise analysis
ahmedabad_check = (
silver[
(silver["City"] == "Ahmedabad") &
(silver["AQI"].notna())
]
.groupby("Year")
.agg(
avg_aqi=("AQI", "mean"),
max_aqi=("AQI", "max"),
days_with_aqi=("AQI", "count")
)
.round({"avg_aqi": 0, "max_aqi": 0})
.reset_index()
.sort_values("Year")
)

display(ahmedabad_check)

ahmedabad_check.to_csv(
"/Volumes/workspace/default/air_quality_data/gold_ahmedabad_yearly.csv",
index=False
)


Year,avg_aqi,max_aqi,days_with_aqi
2015,311.0,1247.0,263
2016,310.0,1842.0,117
2017,559.0,1747.0,69
2018,622.0,2049.0,357
2019,516.0,1719.0,352
2020,242.0,1291.0,176


In [0]:
## All AQI readings above 500

above_500 = (
silver[silver["AQI"] > 500]
.groupby("City")
.agg(
rows_above_500=("AQI", "count")
)
.reset_index()
.sort_values("rows_above_500", ascending=False)
)

display(above_500)

print(
"Total rows above 500:",
(silver["AQI"] > 500).sum()
)

print(
"Total rows with AQI:",
silver["AQI"].notna().sum()
)

above_500.to_csv(
"/Volumes/workspace/default/air_quality_data/gold_aqi_above_500.csv",
index=False
)


City,rows_above_500
Ahmedabad,412
Delhi,48
Patna,25
Gurugram,22
Lucknow,15
Jorapokhar,7
Amritsar,4
Hyderabad,4
Talcher,4
Guwahati,2


Total rows above 500: 543
Total rows with AQI: 24850


In [0]:
# City-wise AQI Analysis
gold_city = (
silver[silver["AQI"].notna()]
.groupby("City")
.agg(
avg_aqi_all=("AQI", "mean"),
max_aqi=("AQI", "max"),
days_with_aqi=("AQI", "count")
)
.reset_index()
)

capped_avg = (
silver[silver["AQI"].notna() & (silver["AQI"] <= 500)]
.groupby("City")["AQI"]
.mean()
.rename("avg_aqi_capped")
)

above_500_count = (
silver[silver["AQI"] > 500]
.groupby("City")["AQI"]
.count()
.rename("rows_above_500")
)

gold_city = (
gold_city
.merge(capped_avg, on="City", how="left")
.merge(above_500_count, on="City", how="left")
)

gold_city["rows_above_500"] = (
gold_city["rows_above_500"].fillna(0).astype(int)
)

gold_city = gold_city.round(
{
"avg_aqi_all": 1,
"avg_aqi_capped": 1,
"max_aqi": 1
}
)

gold_city = gold_city.sort_values(
"avg_aqi_capped",
ascending=False,
na_position="last"
)

display(gold_city)

gold_city.to_csv(
"/Volumes/workspace/default/air_quality_data/gold_city_avg_aqi.csv",
index=False
)


City,avg_aqi_all,max_aqi,days_with_aqi,avg_aqi_capped,rows_above_500
Ahmedabad,452.1,2049.0,1334,291.7,412
Delhi,259.5,716.0,1999,252.1,48
Patna,240.8,619.0,1459,235.6,25
Gurugram,225.1,891.0,1453,219.4,22
Lucknow,218.0,707.0,1893,215.2,15
Talcher,172.9,570.0,698,170.8,4
Jorapokhar,159.3,604.0,771,155.6,7
Brajrajnagar,150.3,355.0,713,150.3,0
Kolkata,140.6,475.0,754,140.6,0
Guwahati,140.1,956.0,495,137.0,2


In [0]:
# Monthly AQI Trend
monthly_data = silver[
silver["AQI"].notna() &
(silver["AQI"] <= 500)
]

gold_monthly = (
monthly_data
.groupby(["Year", "Month", "Season"])
.agg(
avg_aqi=("AQI", "mean"),
readings=("AQI", "count"),
cities_reporting=("City", "nunique")
)
.round({"avg_aqi": 1})
.reset_index()
.sort_values(["Year", "Month"])
)

display(gold_monthly)

gold_monthly.to_csv(
"/Volumes/workspace/default/air_quality_data/gold_monthly_trend.csv",
index=False
)


Year,Month,Season,avg_aqi,readings,cities_reporting
2015,1,Winter,337.8,33,2
2015,2,Winter,328.5,42,2
2015,3,Summer,235.1,75,6
2015,4,Summer,185.8,167,6
2015,5,Summer,187.3,182,6
2015,6,Monsoon,169.1,179,6
2015,7,Monsoon,149.6,184,6
2015,8,Monsoon,156.3,186,6
2015,9,Monsoon,165.1,177,6
2015,10,Post-Monsoon,215.9,191,7


In [0]:
# Pollutant summary 
gold_pollutants = (
silver
.groupby("City")
.agg(
avg_pm2_5=("PM2_5", "mean"),
avg_pm10=("PM10", "mean"),
avg_no2=("NO2", "mean"),
avg_so2=("SO2", "mean"),
avg_co=("CO", "mean"),
avg_o3=("O3", "mean"),
avg_nh3=("NH3", "mean"),
pm2_5_readings=("PM2_5", "count"),
pm10_readings=("PM10", "count")
)
.round(
{
"avg_pm2_5": 1,
"avg_pm10": 1,
"avg_no2": 1,
"avg_so2": 1,
"avg_co": 1,
"avg_o3": 1,
"avg_nh3": 1
}
)
.reset_index()
.sort_values("avg_pm2_5", ascending=False)
)

display(gold_pollutants)

gold_pollutants.to_csv(
"/Volumes/workspace/default/air_quality_data/gold_pollutant_summary.csv",
index=False
)


City,avg_pm2_5,avg_pm10,avg_no2,avg_so2,avg_co,avg_o3,avg_nh3,pm2_5_readings,pm10_readings
Patna,123.5,126.7,37.5,22.1,1.5,37.2,18.4,1537,191
Delhi,117.2,232.8,50.8,15.9,2.0,51.3,42.0,2007,1932
Gurugram,117.1,191.5,23.4,9.4,1.3,34.4,26.1,1525,740
Lucknow,109.7,null,33.2,9.9,2.1,36.9,29.3,1907,0
Ahmedabad,67.9,114.6,59.0,55.3,22.2,39.2,null,1381,407
Kolkata,64.4,115.6,40.4,8.4,0.8,30.7,18.4,759,759
Jorapokhar,64.2,149.7,9.3,33.6,0.7,32.4,7.0,379,894
Brajrajnagar,64.1,124.2,16.5,9.8,1.8,16.6,36.7,753,765
Guwahati,63.7,116.6,13.6,14.7,0.7,25.1,11.1,501,502
Talcher,61.4,165.8,13.8,28.5,1.8,17.2,11.5,741,746


In [0]:
# AQL category analysis

gold_category_days = (
silver[silver["AQI_Bucket"].notna()]
.groupby(["City", "Year", "AQI_Bucket"], observed=True)
.agg(
days=("AQI_Bucket", "count")
)
.reset_index()
.sort_values(["City", "Year", "AQI_Bucket"])
)

display(gold_category_days)

gold_category_days.to_csv(
"/Volumes/workspace/default/air_quality_data/gold_aqi_category_days.csv",
index=False
)


City,Year,AQI_Bucket,days
Ahmedabad,2015,Good,1
Ahmedabad,2015,Moderate,72
Ahmedabad,2015,Poor,74
Ahmedabad,2015,Satisfactory,8
Ahmedabad,2015,Severe,51
Ahmedabad,2015,Very Poor,57
Ahmedabad,2016,Moderate,12
Ahmedabad,2016,Poor,65
Ahmedabad,2016,Satisfactory,3
Ahmedabad,2016,Severe,20


In [0]:
# Total category reading
total_category_readings = gold_category_days["days"].sum()

print("Total category readings:", total_category_readings)




Total category readings: 24850
